# Tactical Fingerprints in Football
**Discovering Team Identities Through Event and Network Analysis**

This notebook walks through the full pipeline interactively:
1. Load & preprocess Wyscout event data
2. Extract tactical features per team-match
3. Build passing networks and compute graph metrics
4. Aggregate into team profiles
5. PCA + UMAP dimensionality reduction
6. Hierarchical clustering → Tactical Fingerprints
7. Evaluate fingerprints against match outcomes

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')

## 1. Setup — Copy Data Files
Place the Wyscout JSON files under `data/raw/`:
```
data/raw/
  events/
    events_England.json
    events_Spain.json
    events_Italy.json
    events_France.json
    events_Germany.json
    events_European_Championship.json
    events_World_Cup.json
  matches/
    matches_England.json  ... (same pattern)
  players.json
  teams.json
  coaches.json
  playerank.json
  eventid2name.csv
  tags2name.csv
```

In [ ]:
from src.data.loader import load_events, load_matches, load_teams, load_players

# Start with one competition for speed; switch to None for all
COMPETITIONS = ['England', 'Spain', 'Germany']

events_df  = load_events(COMPETITIONS)
matches_df = load_matches(COMPETITIONS)
teams_df   = load_teams()
players_df = load_players()

print(f'Events:  {len(events_df):,}')
print(f'Matches: {len(matches_df):,}')
print(f'Teams:   {len(teams_df):,}')
events_df.head()

## 2. Preprocessing

In [ ]:
from src.data.preprocessor import preprocess, filter_passes

events_clean = preprocess(events_df, matches_df)
print(f'Clean events: {len(events_clean):,}')
print('Event type distribution:')
print(events_clean['eventName'].value_counts())

## 3. Event Feature Extraction

In [ ]:
from src.features.event_features import extract_event_features

event_feats = extract_event_features(events_clean)
print(f'Feature rows: {len(event_feats):,}')
event_feats.describe()

## 4. Passing Networks

In [ ]:
from src.network.builder import build_all_networks
from src.network.metrics import compute_network_metrics

passes   = filter_passes(events_clean, accurate_only=True)
networks = build_all_networks(passes)
net_metrics = compute_network_metrics(networks)

print(f'Networks: {len(networks):,}')
net_metrics.describe()

In [ ]:
# Visualise a single team's passing network
import networkx as nx
import matplotlib.pyplot as plt

sample_key = list(networks.keys())[0]
G = networks[sample_key]

fig, ax = plt.subplots(figsize=(8, 6))
weights = [G[u][v]['weight'] for u,v in G.edges()]
pos = nx.spring_layout(G, seed=42)
nx.draw_networkx(G, pos=pos, ax=ax,
                 width=[w*0.3 for w in weights],
                 node_size=300, font_size=7,
                 edge_color='steelblue', node_color='#FF6B35', alpha=0.85)
ax.set_title(f'Passing Network — matchId={sample_key[0]}, teamId={sample_key[1]}')
plt.tight_layout()

## 5. Team Profile Aggregation

In [ ]:
from src.features.aggregator import aggregate_team_profiles, build_fingerprint_matrix

combined = event_feats.merge(net_metrics, on=['matchId', 'teamId'], how='inner')
profiles = aggregate_team_profiles(combined, matches_df)
profiles = profiles.merge(teams_df[['teamId', 'name']], on='teamId', how='left')

print(f'Team profiles: {len(profiles):,}')
profiles[['name', 'competition', 'win_rate', 'points_per_match',
           'pass_accuracy', 'density', 'centralization']].sort_values('win_rate', ascending=False).head(15)

## 6. PCA Analysis

In [ ]:
from src.clustering.fingerprints import scale_features, run_pca, find_optimal_clusters
from src.visualization.plots import plot_pca_variance, plot_silhouette_curve

X, y = build_fingerprint_matrix(profiles)
feature_cols = [c for c in X.columns if c not in {'teamId', 'competition'}]

X_raw = X[feature_cols].fillna(0).values
X_scaled, scaler = scale_features(X_raw)
X_pca, pca = run_pca(X_scaled, n_components=10)

plot_pca_variance(np.cumsum(pca.explained_variance_ratio_))
plt.show()

print('\nTop features in PC1:')
pc1_loadings = pd.Series(pca.components_[0], index=feature_cols).abs().sort_values(ascending=False)
print(pc1_loadings.head(8))

## 7. Optimal Cluster Selection

In [ ]:
eval_df = find_optimal_clusters(X_pca, k_range=range(2, 9))
plot_silhouette_curve(eval_df)
plt.show()
print(eval_df.to_string(index=False))

## 8. Tactical Fingerprints

In [ ]:
from src.clustering.fingerprints import build_fingerprint_report
from src.visualization.plots import (
    plot_cluster_scatter, plot_fingerprint_radar, plot_outcome_by_fingerprint
)

N_CLUSTERS = 5  # Adjust based on silhouette curve above

full_profiles = profiles.merge(
    y[['teamId', 'competition', 'win_rate', 'points_per_match', 'goalDiff']],
    on=['teamId', 'competition'], how='left'
)

result = build_fingerprint_report(
    profiles=full_profiles,
    feature_cols=feature_cols,
    n_clusters=N_CLUSTERS,
    use_umap=True,
)

labelled = result['profiles_labelled']
labelled = labelled.merge(teams_df[['teamId', 'name']], on='teamId', how='left')

plot_cluster_scatter(labelled, teams_df=teams_df)
plt.show()

In [ ]:
# Radar chart of fingerprint styles
radar_features = ['pass_accuracy', 'long_ball_ratio', 'cross_ratio',
                  'smart_pass_ratio', 'density', 'centralization',
                  'pass_att_third', 'counter_rate']
plot_fingerprint_radar(labelled, feature_cols=radar_features)
plt.show()

## 9. Fingerprints vs Match Outcomes

In [ ]:
plot_outcome_by_fingerprint(labelled, metric='win_rate')
plt.show()

plot_outcome_by_fingerprint(labelled, metric='points_per_match')
plt.show()

In [ ]:
# Fingerprint summary table
summary = labelled.groupby('fingerprint').agg(
    n_teams=('teamId', 'nunique'),
    win_rate=('win_rate', 'mean'),
    points_per_match=('points_per_match', 'mean'),
    pass_accuracy=('pass_accuracy', 'mean'),
    long_ball_ratio=('long_ball_ratio', 'mean'),
    cross_ratio=('cross_ratio', 'mean'),
    density=('density', 'mean'),
    centralization=('centralization', 'mean'),
).round(3)

print('\nTactical Fingerprint Summary')
print('=' * 70)
print(summary.to_string())

In [ ]:
# Teams per fingerprint
for fp_id, grp in labelled.groupby('fingerprint'):
    print(f'\n── Fingerprint {fp_id} ({len(grp)} teams) ──')
    print(', '.join(grp['name'].dropna().unique()[:20]))